# nitid Tutorial 1: Inference with Pretrained Models

nitid runs object detection, instance segmentation and semantic segmentation with models from the DETR family. The code is released under Apache 2.0.

This notebook walks through the basic nitid workflow:
1. Set up the project environment
2. Run prediction on a real image with a pretrained model
3. Inspect and save results
4. Other tasks that nitid supports


This notebook runs locally from a nitid checkout, or in Google Colab.

# Set up the project environment

**Local use**: from the repository root, install the dependencies and start Jupyter:
```bash
uv sync
uv run --with jupyterlab jupyter lab
```

**Google Colab**: the setup cell below detects Colab and installs nitid from GitHub.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Installing nitid...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "nitid",
        ],
        check=True,
    )
    TUTORIAL_DIR = Path("/content")
else:

    def find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if all((candidate / p).exists() for p in ["dfine", "tools", "configs"]):
                return candidate
        return start

    TUTORIAL_DIR = find_repo_root(Path.cwd().resolve()) / "tutorials"

os.chdir(TUTORIAL_DIR)
print(f"Running in Colab: {IN_COLAB}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
from nitid import NITID

print("Imported NITID successfully")

# Run prediction on a real image using pre trained AI models
For running a prediction on a real image, we follow these steps:
1) Download a test image
2) Run prediction with Python

In [ ]:
# The image is downloaded at runtime so the repository does not need to store example image files.

from urllib.request import urlretrieve

outputs_dir = TUTORIAL_DIR / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

image_url = "https://ultralytics.com/images/bus.jpg"
image_path = outputs_dir / "bus.jpg"

if not image_path.exists():
    print(f"Downloading {image_url}")
    urlretrieve(image_url, image_path)

print(f"Image ready: {image_path}")

You can replace `model1s` with `model1m`, `model1l`, or `model1x`.

This is the main user-facing workflow: create a model from a supported nitid model name and run detection. nitid downloads, wraps, and loads the matching checkpoint on first use.

In [ ]:
model = NITID("model1s", task="detect")
result = model.predict(str(image_path), conf=0.5)

result = result[0]
detections = result.to_json()

print(result)
print("Detections:", len(result))
print(detections)

# Inspect and save results
`results.save()` writes an annotated image with boxes and labels.

In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

annotated_path = outputs_dir / "bus_result.jpg"
result.save(str(annotated_path))
print(f"Saved annotated image to: {annotated_path}")

if Image is not None and display is not None:
    display(Image(filename=str(annotated_path)))
else:
    print("Install IPython or open the saved file to view the annotated image.")

# Other tasks that nitid supports
The same API covers the other two tasks:
1) Instance segmentation
2) Semantic segmentation (fine-tuning required)

**Segmentation** 
For instance segmentation we follow these steps: 
* Download a test image
* Run a segmentation instance
* Inspect and save results


In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000000139.jpg"
image_path = outputs_dir / "room.jpg"

if not image_path.exists():
    print(f"Downloading {image_url}")
    urlretrieve(image_url, image_path)

print(f"Image ready: {image_path}")

In [ ]:
# For instance segmentation, select the task when constructing the model:
segment_model = NITID("model1s", task="segment")
result = segment_model.predict(str(image_path), conf=0.5)[0]

boxes = result.boxes.xyxy
masks = result.masks.data  # uint8 [N, H, W]
polygons = result.masks.xyn  # normalized polygons

print(result)
print("Boxes:", len(boxes))
print("Mask:", len(masks))
print("Polygons:", len(polygons))

In [ ]:
annotated_path = outputs_dir / "room_result.jpg"
result.save(str(annotated_path))
print(f"Saved annotated image to: {annotated_path}")

if Image is not None and display is not None:
    display(Image(filename=str(annotated_path)))
else:
    print("Install IPython or open the saved file to view the annotated image.")

**Semantic segmentation**

There are no pretrained semantic weights yet. `NITID("model1s", task="semantic")` builds a semantic model whose shared features are initialized from the COCO instance-segmentation checkpoint; its per-pixel head still has to be trained, so predictions before fine-tuning are not meaningful.

To train one on your own masks, see the [fine-tuning guide](../docs/fine_tuning.md) and the training tutorial in this folder.